# 04. ElasticNet Logistic Regression — 해석 가능한 선형 기준선

Logistic Regression은 Feature의 선형 결합을 이탈 확률로 변환하는 강한 기준선입니다. ElasticNet은 L1(일부 계수를 0으로 만드는 희소화)과 L2(상관된 Feature의 계수를 안정화)를 결합합니다. `C`가 작을수록 규제가 강하고, `l1_ratio`가 1에 가까울수록 L1 비중이 커집니다.

이 구현은 범주형을 결측 대체 후 One-Hot Encoding하고, 수치형은 무한값을 NaN으로 바꾼 뒤 중앙값 대체와 StandardScaler를 적용합니다. 모든 단계는 ColumnTransformer와 Pipeline 안에 있어 각 Fold의 학습 데이터에만 `fit`되며, 검증 Fold 정보 누수를 막습니다. 희소 출력을 유지해 895,005행에서 메모리 폭증을 피합니다.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd() if (Path.cwd() / 'models').is_dir() else Path.cwd().parent
models_dir = repo_root / 'models'
if str(models_dir) not in sys.path:
    sys.path.insert(0, str(models_dir))
model_path = models_dir / '04_elasticnet_logistic_weekly_next_week.py'
spec = spec_from_file_location('elasticnet_weekly_model', model_path)
elasticnet_model = module_from_spec(spec)
assert spec and spec.loader
sys.modules[spec.name] = elasticnet_model
spec.loader.exec_module(elasticnet_model)

## 제한 후보와 불균형

전체 데이터에서 대규모 Grid Search 대신 두 후보만 비교합니다. `class_weight=None`은 원시 확률 보정에 유리한 반면, `balanced`는 희소한 이탈 클래스의 손실을 크게 올려 Recall을 높일 수 있지만 확률을 심하게 과대 추정할 수 있습니다. 최종 선택은 전체 OOF **PR-AUC 최대**를 우선하며, 동률이면 Brier Score와 ECE가 낮은 후보를 택합니다.

SAGA의 `ConvergenceWarning`은 설정된 반복 안에 정지 조건을 만족하지 못했다는 뜻입니다. Fold별 경고 여부와 `n_iter`를 기록해 단순히 결과값만 제출하지 않습니다. 순위 지표가 좋아도 Brier/ECE가 나쁘면 확률 자체는 개입 우선순위 외의 의사결정에 부적합할 수 있습니다.

In [ ]:
# 전체 후보 비교를 다시 실행할 때만 True로 바꿉니다.
RUN_TRAINING = False
output_dir = repo_root / 'models' / 'demo_1'
if RUN_TRAINING:
    data_path = elasticnet_model.resolve_data_path(None)
    result = elasticnet_model.run_elasticnet(data_path, output_dir=output_dir)
    metrics = result['metrics']
    fold_metrics = result['fold_metrics']
    candidates = result['candidate_metrics']
else:
    metrics = pd.read_csv(output_dir / 'elasticnet_logistic_weekly_next_week_metrics.csv')
    fold_metrics = pd.read_csv(output_dir / 'elasticnet_logistic_weekly_next_week_fold_metrics.csv')
    candidates = pd.read_csv(output_dir / 'elasticnet_logistic_candidate_metrics.csv')
display(candidates)
display(metrics)
display(fold_metrics)

## Dummy 대비 개선 확인

PR-AUC와 Recall@Top-20%는 순위 성능을, Brier와 ECE는 확률 품질을 비교합니다. Dummy의 Recall은 확률 동률 때문에 참고값이라는 점을 유지하고, ElasticNet의 평균 확률이 실제 양성률보다 큰지도 함께 확인합니다.

In [ ]:
dummy_metrics = pd.read_csv(output_dir / 'dummy_weekly_next_week_metrics.csv')
comparison_columns = [
    'model', 'target_rate', 'recall_at_top_20pct',
    'pr_auc', 'brier_score', 'ece_10bin'
]
comparison = pd.concat([dummy_metrics, metrics], ignore_index=True)
display(comparison[comparison_columns])